In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import nltk
import re

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess(text):
    if not isinstance(text, str):
        return ""

    try:
        text = text.lower()
        text = re.sub(r"http\S+|[^a-z\s]", "", text)
        tokens = nltk.word_tokenize(text)  # make sure this is correct!
        tokens = [lemmatizer.lemmatize(stemmer.stem(word)) for word in tokens if word not in stop_words]
        return " ".join(tokens)
    except Exception as e:
        print(f"Error in preprocessing: {text}\n{e}")
        return ""


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
df = pd.read_csv('dataset/fake-news-dataset.csv')  # Make sure this file is in your working dir
df = df[['text', 'lebel']].dropna()
df['text'] = df['text'].fillna('').astype(str)
df['clean_text'] = df['text'].apply(preprocess)
df['label'] = df['lebel'].astype(int)
df.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_13576\3325118759.py:1: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset/final_en.csv')  # Make sure this file is in your working dir


ValueError: invalid literal for int() with base 10: ' it seems to have collapsed under the weight of its own mountain of digital ash. Maybe that was the idea. READ MORE MSM LIES AT: 21st Century Wire MSM FilesSUPPORT 21WIRE  SUBSCRIBE NOW & BECOME A ME

In [21]:
df = pd.read_csv('dataset/final_en.csv')
X_train,x_text,ytrain,ytest=train_test_split(df['text'],df['lebel'],test_size=0.2,random_state=42)
test_df=pd.concat([x_text,ytest],axis=1)
test_df.to_csv("dataset/test_dataset.csv",index=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_13576\1238417359.py:1: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset/final_en.csv')


In [22]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['label'], test_size=0.2, random_state=42)

train_dataset = FakeNewsDataset(X_train, y_train)
test_dataset = FakeNewsDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

KeyError: 'clean_text'

In [34]:
class RobertaWithFFNN(nn.Module):
    def __init__(self):
        super(RobertaWithFFNN, self).__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(0.3)
        self.ffnn = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)  # For binary classification
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        x = self.dropout(cls_output)
        return self.ffnn(x)

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RobertaWithFFNN().to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
model.train()
epochs = 2

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/2, Loss: 0.0729
Epoch 2/2, Loss: 0.0250


In [25]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds))

NameError: name 'model' is not defined

In [3]:
import torch.nn.functional as f

Load the trained tokenizer 

In [26]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained('roberta_tokenizer')


load the model

In [27]:

# Define the model class exactly as before
class RobertaWithFFNN(nn.Module):
    def _init_(self):
        super(RobertaWithFFNN, self)._init_()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(0.3)
        self.ffnn = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        return self.ffnn(x)

# Load full model
model = torch.load("model/roberta_fakenews_ffnn.pth", map_location=torch.device("cpu"))
model.eval()

C:\Users\Admin\AppData\Local\Temp\ipykernel_13576\3304656114.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("model/roberta_fakenews_ffnn.pth", map_l

RobertaWithFFNN(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm):

In [28]:
def predict_label(text):
    model.eval()
    text = preprocess(text)  # same cleaning as training
    encoded = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        prediction = torch.argmax( output, dim=1).item()

   
    return "Fake" if prediction==0 else "Genuine"

In [36]:
sample_text = "OTTAWA (Reuters) - Canadian Prime Minister Justin Trudeau s government has been badly shaken by a conflict-of-interest controversy about his finance minister, but the Liberal government s upcoming fiscal update offers an opportunity to reset the public focus on Canada s strong economy, political observers say. The government has been plagued all week about questions about the finance minister, Bill Morneau, the multimillionaire former chief executive officer of human resources management firm Morneau Shepell. Some have questioned whether Morneau would be forced to resign. The focus has been a rare stumble for Trudeau s government, which marked two years in office this month and has mostly maneuvered its way out of political trouble partly because of Trudeau s personal popularity and the youthful momentum of the Liberals after 10 years of Conservative rule.  They re politically very astute in a whole bunch of areas but in issues management and parliamentary management they seem to be ham-fisted,  said Andrew Graham, professor at the school of policy studies at Queen s University. With Trudeau s strong defense of Morneau, and the finance minister likely to unveil a smaller budget deficit in Tuesday s fall fiscal update, Liberals have a temporary chance to refocus on good news.  Economic conditions are a bedrock of whether people feel good or bad about how politicians are performing, and the economy is doing very well for most people,  said Abacus Data pollster Bruce Anderson.  Morneau said on Thursday he will put his assets in a blind trust and divest stock in a publicly traded family business. That comes after weeks of backlash over tax reform that has become a major obstacle for Trudeau s government. Opposition parties from both the political left and right have seized on the ethics scandal, trying to tie Trudeau s team to what they say is an entitled Liberal Party that has previously faced corruption charges.   The opposition has changed the focus from substance to ethics, and they won t let that go that easily,  said Genevieve Tellier, a political professor at the University of Ottawa. But Tellier said Trudeau s decision to fill his cabinet with political rookies - including Morneau - rather than turning to the old Liberal guard, could limit the ability of the opposition to land many ethical punches. Moreover, Morneau is respected by markets.  I think the prime minister would be very cautious about changing his finance minister. ... (Morneau) presents a reassuring image, he doesn t scare the markets,  Tellier said. Ipsos Public Affairs pollster Darrell Bricker put it more bluntly:  They really don t have a choice but to tough it out and hope some event will transpire to distract the hyenas.  The expected budget improvement in Tuesday s fall fiscal update could also give Morneau the leeway to woo voters with more spending or debt reduction.  The way the numbers are playing out, they are in a fairly favorable fiscal position,  said Paul Ferley, assistant chief economist at Royal Bank of Canada. Morneau spokesman Dan Lauzon said the finance minister has no plans to change his strategy, and would keep focus on fiscal stimulus matters:  He is in this for the long run, and he won t let distractions get in the way.  ."
print(predict_label(sample_text))

Genuine


In [9]:
# Save the entire model
torch.save(model, "roberta_fakenews_ffnn.pth")

# Save the tokenizer
tokenizer.save_pretrained("roberta_tokenizer")

('roberta_tokenizer\\tokenizer_config.json',
 'roberta_tokenizer\\special_tokens_map.json',
 'roberta_tokenizer\\vocab.json',
 'roberta_tokenizer\\merges.txt',
 'roberta_tokenizer\\added_tokens.json')